# Aula 1: Árvores de Decisão
Classificação com árvores de decisão

Árvores de decisão são modelos de aprendizado supervisionado que realizam classificação ao dividir recursivamente os dados com base nas características (features), gerando uma estrutura em forma de árvore onde cada nó interno representa uma decisão sobre um atributo, cada ramo representa o resultado dessa decisão e cada folha representa uma classe ou rótulo. Para classificação, o algoritmo busca separar as classes maximizando a pureza dos nós, utilizando critérios como Gini ou entropia para determinar as melhores divisões.

In [ ]:
# -*- coding: utf-8 -*-

# Resolvendo um problema de Classificação
import os
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

import pandas as pd



In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vqrca/ml-datasets/refs/heads/main/Heart_Disease_Prediction.csv')



In [ ]:
pd.set_option('display.max_columns', None)
print(df.head())



In [ ]:
df.shape



In [ ]:
df.info()



Muitos algoritmos de machine learning exigem dados numéricos. Variáveis categóricas precisam ser codificadas antes de serem utilizadas. A técnica de one-hot encoding (get_dummies) converte variáveis categóricas em colunas binárias (0 ou 1), onde cada categoria se torna uma coluna separada. Isso evita que o modelo interprete categorias como valores ordinais com relações de maior/menor que não existem.

In [ ]:
# Aplicação do get_dummies nas colunas Chest pain type e Thallium

df_encoded = pd.get_dummies(df, columns=['Chest pain type', 'Thallium'], dtype=int)

df_encoded.head()



Para avaliar a capacidade de generalização de um modelo, os dados são divididos em dois conjuntos: treino (usado para treinar o modelo) e teste (usado para avaliar). A separação típica utiliza 80% dos dados para treino e 20% para teste. O parâmetro stratify garante que a proporção das classes seja mantida em ambos os conjuntos, e random_state assegura reprodutibilidade dos resultados.

In [ ]:
X = df_encoded.drop('Heart Disease', axis=1)
y = df_encoded['Heart Disease']



In [ ]:
X



In [ ]:
y



In [ ]:
from sklearn.model_selection import train_test_split
X_treino, X_teste, y_treino, y_teste = train_test_split(X,
                                                        y, 
                                                        test_size=0.2,
                                                        stratify=y,
                                                        random_state=42)



In [ ]:
from sklearn.tree import DecisionTreeClassifier



In [ ]:
# Cria uma instância do Decision tree: dt
modelo_arvore = DecisionTreeClassifier(random_state=5389)

# Ajusta o classificador ao conjunto de treinamento
modelo_arvore.fit(X_treino, y_treino)

# Preve o Target do conjunto de teste
predicoes = modelo_arvore.predict(X_teste)



In [ ]:
predicoes



In [ ]:
# Calcula as probabilidades para cada classe
probabilidades = modelo_arvore.predict_proba(X_teste)

probabilidades



In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(24, 12))

plot_tree(
    modelo_arvore,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    #max_depth=3,
    fontsize=8,
    proportion=True,
    precision=2
)

plt.tight_layout()
plt.show()



A acurácia mede a proporção de previsões corretas, mas pode ser enganosa em datasets imbalanced. O relatório de classificação fornece precisão (precision), recall e F1-score para cada classe. A matriz de confusão mostra visualmente verdadeiros positivos, falsos positivos, verdadeiros negativos e falsos negativos. A curva ROC (Receiver Operating Characteristic) plota a taxa de verdadeiros positivos contra a taxa de falsos positivos para diferentes limiares, e a área sob a curva (AUC) resume a capacidade discriminatória do modelo.

In [ ]:
from sklearn.metrics import accuracy_score



In [ ]:
acuracia_teste = accuracy_score(y_teste, predicoes)
print(f'Acurácia nos dados de teste: {acuracia_teste:.2%}')



In [ ]:
predicoes_treino = modelo_arvore.predict(X_treino)

acuracia_treino= accuracy_score(y_treino, predicoes_treino)
print(f'Acurácia nos dados de treino: {acuracia_treino:.2%}')



In [ ]:
from sklearn.metrics import classification_report



In [ ]:
# Gera um relatório de classificação
relatorio = classification_report(y_teste, predicoes)

print(relatorio)



In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay



In [ ]:
ConfusionMatrixDisplay.from_estimator(modelo_arvore, X_teste, y_teste);



In [ ]:
ConfusionMatrixDisplay.from_estimator(modelo_arvore, X_teste, y_teste,
                                      normalize = 'true',
                                      cmap = 'Blues');



In [ ]:
from sklearn.metrics import RocCurveDisplay



In [ ]:
# Gera a curva ROC
RocCurveDisplay.from_estimator(
    modelo_arvore,
    X_teste,
    y_teste
);



Overfitting ocorre quando o modelo aprende os dados de treino excessivamente bem, incluindo ruído, resultando em baixa performance em dados novos. A poda (pruning) controla a complexidade da árvore limitando sua profundidade (max_depth) ou o número mínimo de amostras em folhas (min_samples_leaf). Uma árvore muito profunda memoriza os dados; uma árvore muito simples pode subajustar. O equilíbrio entre viés e variância é essencial para modelos generalizáveis.

In [ ]:
# Cria uma instância do Decision tree: dt
modelo_arvore_podada = DecisionTreeClassifier(random_state=5389, max_depth=3)

# Ajusta o classificador ao conjunto de treinamento
modelo_arvore_podada.fit(X_treino, y_treino)

# Preve o Target do conjunto de teste
predicoes_podada = modelo_arvore_podada.predict(X_teste)



In [ ]:
# Preve o Target do conjunto de treinamento
predicoes_podada_treino = modelo_arvore_podada.predict(X_treino)

# Calcula a acurácia no conjunto de treinamento
acuracia_treino_podada = accuracy_score(y_treino, predicoes_podada_treino)
print(f'Acurácia nos dados de treino (max_depth=3): {acuracia_treino_podada:.2f}')

# Acurácia no conjunto de teste 
acuracia_teste_podada = accuracy_score(y_teste, predicoes_podada)
print(f'Acurácia nos dados de teste (max_depth=3): {acuracia_teste_podada:.2f}')



In [ ]:
plt.figure(figsize=(24, 12))

plot_tree(
    modelo_arvore_podada,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    proportion=True,
    precision=2
)

plt.tight_layout()
plt.show()



In [ ]:
ConfusionMatrixDisplay.from_estimator(modelo_arvore_podada, X_teste, y_teste,
                                      normalize = 'true',
                                      cmap = 'Blues');



A validação cruzada (cross-validation) avalia a performance do modelo dividindo os dados em k partições (folds) e treinando/testando k vezes, cada vez usando uma partição diferente para teste e as restantes para treino. A validação cruzada estratificada (StratifiedKFold) mantém a proporção das classes em cada fold. A função cross_validate permite calcular múltiplas métricas simultaneamente, fornecendo uma avaliação mais robusta e confiável.

In [ ]:
from sklearn.model_selection import cross_val_score



In [ ]:
modelo = DecisionTreeClassifier(random_state=5389, max_depth=3)
scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
scores



In [ ]:
print("O resultado da validação cruzada é %0.2f%% acurácia com desvio padrão de %0.2f%%" 
      % (scores.mean() * 100, scores.std() * 100))



In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score



In [ ]:
validacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=5389
)

scores = cross_val_score(
    modelo_arvore_podada,
    X,
    y,
    cv=validacao,
    scoring='accuracy'
)

scores



In [ ]:
print("O resultado da validação cruzada é %0.2f%% acurácia com desvio padrão de %0.2f%%" 
      % (scores.mean() * 100, scores.std() * 100))



In [ ]:
from sklearn.model_selection import cross_validate



In [ ]:
resultado = cross_validate(
    modelo_arvore_podada,
    X,
    y,
    cv=5,
    scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
)

resultado



In [ ]:
resultados_metricas = pd.DataFrame({
    'Média': [
        resultado['test_accuracy'].mean(),
        resultado['test_precision_macro'].mean(),
        resultado['test_recall_macro'].mean(),
        resultado['test_f1_macro'].mean()
    ],
    'Desvio Padrão': [
        resultado['test_accuracy'].std(),
        resultado['test_precision_macro'].std(),
        resultado['test_recall_macro'].std(),
        resultado['test_f1_macro'].std()
    ]
},
index=['Acurácia', 'Precisão', 'Recall', 'F1-Score'])

# Aplicar formatação de porcentagem com duas casas decimais
resultados_metricas['Média'] = resultados_metricas['Média'].apply(lambda x: f'{x:.2%}')
resultados_metricas['Desvio Padrão'] = resultados_metricas['Desvio Padrão'].apply(lambda x: f'{x:.2%}')

resultados_metricas



Hiperparâmetros são configurações definidas antes do treinamento (como max_depth, min_samples_leaf, criterion). O GridSearchCV testa todas as combinações de uma grade de hiperparâmetros usando validação cruzada e seleciona a combinação com melhor pontuação. Embora computacionalmente custoso, garante a exploração sistemática do espaço de hiperparâmetros.

In [ ]:
from sklearn.model_selection import GridSearchCV



In [ ]:
# Define a grade de hiperparâmetros que será testada
grade_parametros = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

# Cria uma instância da Árvore de Decisão
modelo_arvore_grid = DecisionTreeClassifier(random_state=5389)

# Configura o Grid Search
# Utiliza a acurácia como métrica de avaliação e validação cruzada com 5 partições
grid_search = GridSearchCV(
    estimator=modelo_arvore_grid,
    param_grid=grade_parametros,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Executa a busca pelos melhores hiperparâmetros
grid_search.fit(X, y)

# Exibe os melhores hiperparâmetros encontrados e a melhor acurácia média
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor acurácia média na validação cruzada: {grid_search.best_score_:.2f}")



In [ ]:
grid_search.best_estimator_



In [ ]:
colunas_resultado = [
    'param_max_depth',
    'param_min_samples_leaf',
    'param_criterion',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]

resultados = (
    pd.DataFrame(grid_search.cv_results_)[colunas_resultado]
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)

resultados



In [ ]:
# Criar uma instância do Decision tree
modelo_arvore_otimizada = DecisionTreeClassifier(random_state=5389, criterion='entropy', max_depth=3, min_samples_leaf=10)

# Ajustar o classificador ao conjunto de treinamento
modelo_arvore_otimizada.fit(X_treino, y_treino)

# Prever o Target do conjunto de teste
predicoes_otimizada = modelo_arvore_otimizada.predict(X_teste)



In [ ]:
plt.figure(figsize=(24, 12))

plot_tree(
    modelo_arvore_otimizada,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    proportion=True,
    precision=2
)

plt.tight_layout()
plt.show()



In [ ]:
# Prever o Target do conjunto de treinamento
predicoes_treino_otimizada = modelo_arvore_otimizada.predict(X_treino)

# Calcular a acurácia no conjunto de treinamento
acuracia_treino_otimizada = accuracy_score(y_treino, predicoes_treino_otimizada )
print(f'Acurácia nos dados de treino (árvore ajustada): {acuracia_treino_otimizada:.2%}')

# Acurácia no conjunto de teste
acuracia_teste_otimizada  = accuracy_score(y_teste, predicoes_otimizada)
print(f'Acurácia nos dados de teste (árvore ajustada): {acuracia_teste_otimizada:.2%}')



In [ ]:
ConfusionMatrixDisplay.from_estimator(modelo_arvore_otimizada,
                                      X_teste, y_teste,
                                      normalize = 'true',
                                      cmap = 'Blues');



In [ ]:
ax = RocCurveDisplay.from_estimator(
    modelo_arvore,
    X_teste,
    y_teste,
    name='Árvore inicial',
    pos_label='Presence'
).ax_

RocCurveDisplay.from_estimator(
    modelo_arvore_podada,
    X_teste,
    y_teste,
    name='Árvore podada',
    pos_label='Presence',
    ax=ax
)

RocCurveDisplay.from_estimator(
    modelo_arvore_otimizada,
    X_teste,
    y_teste,
    name='Árvore otimizada',
    pos_label='Presence',
    ax=ax
)

plt.plot([0, 1], [0, 1], '--', color='gray')
plt.legend()
plt.show()

